In [1]:
import dolfinx
print(dolfinx.__version__)

[1790446212.900328] [DESKTOP-JTCK1QB:939  :0]         netlink.c:134  UCX  ERROR   failed to send netlink message on fd=92: Input/output error
[1790446212.900413] [DESKTOP-JTCK1QB:939  :0]         netlink.c:134  UCX  ERROR   failed to send netlink message on fd=92: Input/output error
0.11.0


In [10]:
import numpy as np
import ufl
from mpi4py import MPI
from slepc4py import SLEPc
from dolfinx import mesh,fem, default_scalar_type
from dolfinx.fem import petsc

In [11]:
L,W,H = 1.0,0.01,0.01
domain = mesh.create_box(MPI.COMM_WORLD, [[0.0, 0.0, 0.0], [L, W, H]], [100, 2, 2])
V = fem.functionspace(domain,("Lagrange",2, (domain.geometry.dim,)))

E = 2e11
nu = 0.3
rho = 7800

lambda_ = E * nu /((1 + nu) * (1 - 2*nu)) if nu != 0.0 else 0.0
mu = E / (2 * (1 + nu))

def epsilon(u):
    return ufl.sym(ufl.grad(u))

def sigma(u):
    return lambda_ * ufl.nabla_div(u) * ufl.Identity(len(u)) + 2 * mu * epsilon(u)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

k_form = fem.form(ufl.inner(sigma(u), epsilon(v)) * ufl.dx)
m_form = fem.form(rho * ufl.inner(u,v) * ufl.dx)

def clamped_boundary(x):
    return np.isclose(x[0],0.0)

fdim = domain.topology.dim - 1
boundary_facets = mesh.locate_entities_boundary(domain,fdim,clamped_boundary)
u_D = np.array([0.0,0.0,0.0], dtype=default_scalar_type)
bc = fem.dirichletbc(u_D,fem.locate_dofs_topological(V,fdim,boundary_facets),V)


In [15]:
K = petsc.assemble_matrix(k_form, bcs=[bc])
K.assemble()
M = petsc.assemble_matrix(m_form, bcs=[bc])
M.assemble()

In [17]:
# # solving the eigenvalue problem
from slepc4py import SLEPc

eps = SLEPc.EPS().create(mesh.comm)

eps.setOperators(K, M)

eps.setProblemType(
    SLEPc.EPS.ProblemType.GHEP
)

eps.setWhichEigenpairs(
    SLEPc.EPS.Which.SMALLEST_REAL
)

eps.setDimensions(nev=10)

eps.solve()


nconv = eps.getConverged()

print("Converged eigenvalues:", nconv)


for i in range(nconv):

    eigenvalue = eps.getEigenvalue(i)

    omega = eigenvalue ** 0.5

    frequency = omega / (2 * 3.141592653589793)

    print(
        "Mode:",
        i + 1,
        "frequency:",
        frequency,
        "Hz"
    )




with dolfinx.io.XDMFFile(mesh.comm, "modes.xdmf", "w") as file:
    file.write_mesh(mesh)

    for i in range(nconv):

        # Create function for this mode
        mode = fem.Function(V)
        mode.name = f"Mode_{i+1}"

        # Get eigenvector
        eps.getEigenvector(i, mode.x.petsc_vec)
        mode.x.scatter_forward()

        # Normalize only for visualization
        max_value = np.max(np.abs(mode.x.array))
        if max_value > 0:
            mode.x.array[:] /= max_value

        # Frequency
        eigenvalue = eps.getEigenvalue(i)
        omega = np.sqrt(eigenvalue.real)
        frequency = omega / (2*np.pi)

        print(f"Mode {i+1}: {frequency:.4f} Hz")

        # Save mode
        file.write_function(mode)

nconv = eps.getConverged()
f_numerical = None

for i in range(nconv):
    val = eps.getEigenvalue(i)
    if val.real > 1e-3:
        f_numerical = np.sqrt(val.real) / (2 * np.pi)
        break

I_z = (W * H**3) / 12.0
Area = W * H
beta_1 = 1.875104

omega_analytical = (beta_1**2 / L**2) * np.sqrt((E * I_z) / (rho * Area))   
f_analytical = omega_analytical / (2 * np.pi)

print(f"--- Natural Frequency Comparison (Mode 1) ---")
if f_numerical:
    print(f"3D DOLFINx FEA (Quadratic, nu=0): {f_numerical:.3f} Hz")
print(f"1D Euler-Bernoulli Theory:        {f_analytical:.3f} Hz")



Converged eigenvalues: 0


AttributeError: module 'dolfinx.mesh' has no attribute 'comm'